# Buddy Editing Planner — LoRA training (Stage B1)
Settings → Accelerator → **GPU T4 x1** (or P100), Internet **on**. Run all. Outputs land in `/kaggle/working/` — download `buddy_planner_outputs.zip` and put its contents in `brain/incoming/` so the result can be verified independently.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
!git clone --depth 1 https://github.com/kopanathi642/YOUTUBE-BUDDY.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install 'transformers>=4.51' 'peft>=0.13' accelerate safetensors pydantic
!python brain/data/build_synth.py   # rebuilds synth_v0 deterministically; hashes must match MANIFEST.json
!git diff --stat brain/data/synth_v0/MANIFEST.json

## Stage A — zero-shot baseline (no training)

In [ ]:
!python brain/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --chat --n 200 --out /kaggle/working/baseline_qwen05.json

## Stage B1 — smoke test (20 steps), then the real run

In [ ]:
!python brain/train_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --smoke

In [ ]:
!python brain/train_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --epochs 2 --out /kaggle/working/planner-qwen05-v1 2>&1 | tee /kaggle/working/train.log

## Evaluate on held-out test + fresh-process verification

In [ ]:
!python brain/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter /kaggle/working/planner-qwen05-v1 --chat --n 400
!python brain/verify.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter /kaggle/working/planner-qwen05-v1 --chat --n 50

## Optional: Qwen3-4B-Instruct-2507 QLoRA (the Director base) — needs ~16 GB GPU, 2–4 h

In [ ]:
# !pip -q install bitsandbytes
# !python brain/train_lora.py --base Qwen/Qwen3-4B-Instruct-2507 --qlora --bs 4 --grad_accum 4 --epochs 1 --out /kaggle/working/planner-qwen3-4b-v1

In [ ]:
!cd /kaggle/working && zip -r buddy_planner_outputs.zip planner-qwen05-v1 baseline_qwen05.json train.log && ls -la